# Weights and Activations

Inspect what a feedforward network is doing internally: parameter distributions, hidden activations, and prediction confidence.

## Initialize PyTorch

- Import PyTorch, torchvision, and plotting tools
- Seed PyTorch for reproducible results
- Detect device (GPU, Apple Silicon, CPU)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

## Load the Dataset

- Load MNIST images and labels
- Normalize images with MNIST mean and standard deviation
- Use a training subset and test subset for quick iteration
- Create `DataLoader`s for batches

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(8000)), batch_size=128, shuffle=True)
test_loader = DataLoader(Subset(test_data, range(512)), batch_size=128)

## Define Model

- Build a feedforward classifier with named layers
- Use named layers so hooks can inspect internal activations
- Output `10` digit class scores
- Prepare loss and optimizer for training

In [ ]:
class MNISTFeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.fc3(x)

model = MNISTFeedForward().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Train Model

- Train the model for a few epochs
- Run forward pass, loss calculation, backpropagation, and optimizer updates
- Create trained weights and activations for later inspection

In [ ]:
for epoch in range(3):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

## Visualize Weights

- Plot histograms of learned weights in each linear layer
- Compare how parameter distributions differ across layers
- Use weight distributions to discuss scale and training dynamics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
layers = [("fc1", model.fc1), ("fc2", model.fc2), ("fc3", model.fc3)]
for ax, (name, layer) in zip(axes, layers):
    ax.hist(layer.weight.detach().cpu().flatten(), bins=40)
    ax.set_title(f"{name} weights")
plt.tight_layout()

## Capture Activations

- Register forward hooks on hidden activation layers
- Run one test batch through the model
- Store hidden activations for later plots
- Remove hooks after collecting activations

In [ ]:
activations = {}

def save_activation(name):
    def hook(module, inputs, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = [
    model.relu1.register_forward_hook(save_activation("hidden_1")),
    model.relu2.register_forward_hook(save_activation("hidden_2")),
]

images, labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    logits = model(images.to(device))
    probabilities = torch.softmax(logits, dim=1).cpu()

for hook in hooks:
    hook.remove()

## Visualize Activations

- Plot hidden activation distributions
- Look for sparse or saturated activations
- Connect activation patterns to how information flows through the network

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].hist(activations["hidden_1"].flatten(), bins=50)
axes[0].set_title("hidden layer 1 activations")
axes[1].hist(activations["hidden_2"].flatten(), bins=50)
axes[1].set_title("hidden layer 2 activations")
plt.tight_layout()

## Visualize Prediction Confidence

- Select one test image
- Display the image and its true label
- Plot model probability for each digit class
- Use confidence scores to discuss uncertainty

In [ ]:
sample_index = 0
plt.figure(figsize=(7, 3))
plt.subplot(1, 2, 1)
plt.imshow(images[sample_index].squeeze(), cmap="gray")
plt.title(f"true label: {labels[sample_index].item()}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(range(10), probabilities[sample_index])
plt.xticks(range(10))
plt.ylim(0, 1)
plt.title("prediction confidence")
plt.tight_layout()